# Supplemental ViT Inference

Runs batch inference for a supplemental ViT checkpoint against a matching RRUFF benchmark HDF5 and reports top-1/3/5 accuracy. No W&B login required.

## Environment

Use the shared train/eval environment:

```bash
conda activate paper-ai-diffraction-train-eval
pip install -e .
```

The following packages are required (already present in `environment-train-eval.yml`):
- `torch`, `h5py`, `numpy`, `tqdm`, `pandas`

## Checkpoints

Download supplemental ViT checkpoints from Zenodo and place them under:
```
external/checkpoints/
```
See `reproducibility/checkpoint_manifest.csv` for the full list.

## RRUFF benchmark HDF5

Large benchmark files reside on Stampede3 scratch. See `reproducibility/dataset_manifest.csv` for paths and the matching checkpoint ↔ benchmark pairings.

In [ ]:
import os
from pathlib import Path

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'configs').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'Could not locate repo root from {cwd}')

MODELS_DIR = REPO_ROOT / 'external' / 'checkpoints' / 'supplemental' / 'Models'

# --- Select checkpoint and matching benchmark ---
# Uncomment the preset you want to evaluate.
# Each entry: (checkpoint filename, RRUFF HDF5 path, spec_length, use_rope, two_theta_min, two_theta_max)

PRESETS = {
    # Tables S4/S5, Fig S3
    'smqmqi14': (
        'xrd_model_smqmqi14.pth',
        '/scratch/10486/dchansew/Data/RRUFF_norm_reduced_range_10_80_deg_701_intensity.hdf5',
        701, False, 10.0, 80.0,
    ),
    # Tables S3/S5
    'mth7zg2w': (
        'xrd_model_mth7zg2w.pth',
        '/scratch/10486/dchansew/Data/RRUFF_norm_uniform_sampling_5_90_deg_1000_intensity.hdf5',
        1000, False, 5.0, 90.0,
    ),
    # Tables S4/S5
    'bj1m83oz': (
        'xrd_model_bj1m83oz.pth',
        '/scratch/10486/dchansew/Data/RRUFF_norm_reduced_range_10_80_deg_700_intensity.hdf5',
        700, False, 10.0, 80.0,
    ),
    # Tables S3/S5 (use_rope=True)
    '3zmiyil8': (
        'xrd_model_3zmiyil8.pth',
        '/scratch/10486/dchansew/Data/RRUFF_norm_uniform_sampling_5_90_deg_1000_intensity.hdf5',
        1000, True, 5.0, 90.0,
    ),
    # Table S3
    'f3sdux88': (
        'xrd_model_f3sdux88.pth',
        '/scratch/10486/dchansew/Data/RRUFF_norm_uniform_sampling_5_90_deg_1000_intensity.hdf5',
        1000, False, 5.0, 90.0,
    ),
    # Table S3
    'kd1znx23': (
        'xrd_model_kd1znx23.pth',
        '/scratch/10486/dchansew/Data/RRUFF_norm_uniform_sampling_5_90_deg_1000_intensity.hdf5',
        1000, False, 5.0, 90.0,
    ),
    # Table S7
    'tite66yo': (
        'xrd_model_tite66yo.pth',
        '/scratch/10486/dchansew/Data/RRUFF_low_bkg_full_intensity.hdf5',
        8501, False, 5.0, 90.0,
    ),
    'k0it6fz2': (
        'xrd_model_k0it6fz2.pth',
        '/scratch/10486/dchansew/Data/RRUFF_low_bkg_full_intensity.hdf5',
        8501, False, 5.0, 90.0,
    ),
    'l0yy9wv9': (
        'xrd_model_l0yy9wv9.pth',
        '/scratch/10486/dchansew/Data/RRUFF_low_bkg_full_intensity.hdf5',
        8501, False, 5.0, 90.0,
    ),
    'cpv8k8ha': (
        'xrd_model_cpv8k8ha.pth',
        '/scratch/10486/dchansew/Data/RRUFF_low_bkg_full_intensity.hdf5',
        8501, False, 5.0, 90.0,
    ),
    # Fig S5 attention
    'pi7r8pah': (
        'xrd_model_pi7r8pah.pth',
        '/scratch/10486/dchansew/Data/RRUFF_low_bkg_full_intensity.hdf5',
        8501, False, 5.0, 90.0,
    ),
}

# Set the run ID to evaluate:
RUN_ID = 'smqmqi14'

ckpt_file, rruff_h5, SPEC_LENGTH, USE_ROPE, TWO_THETA_MIN, TWO_THETA_MAX = PRESETS[RUN_ID]
CHECKPOINT = str(MODELS_DIR / ckpt_file)
RRUFF_H5 = rruff_h5

print(f'Checkpoint : {CHECKPOINT}')
print(f'Benchmark  : {RRUFF_H5}')
print(f'spec_length: {SPEC_LENGTH}')
print(f'use_rope   : {USE_ROPE}')
print(f'2θ range   : {TWO_THETA_MIN}–{TWO_THETA_MAX}°')

In [ ]:
# Model architecture — same for all supplemental ViT checkpoints
MODEL_CONFIG = dict(
    spec_length    = SPEC_LENGTH,
    num_output     = 99,          # categorical 99-class crystal system
    patch_size     = 25,
    embed_dim      = 128,
    depth          = 4,
    num_heads      = 4,
    mlp_ratio      = 4.0,
    drop_ratio     = 0.0,
    use_rope       = USE_ROPE,
    use_mlp_head   = False,
    mlp_head_hidden_dim = 256,
)

NUM_CLASSES  = 99
BATCH_SIZE   = 128
NUM_WORKERS  = 4
# start/end columns: 1-based, inclusive; matches spec_length columns in the HDF5
START_COL    = 1
END_COL      = SPEC_LENGTH

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / 'src'))

import torch
import numpy as np
from collections import OrderedDict
from tqdm.auto import tqdm

from paper_ai_diffraction.core.model import VIT_model
from paper_ai_diffraction.core.dataset import get_dataloaders_test

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = VIT_model(**MODEL_CONFIG)

ckpt = torch.load(CHECKPOINT, map_location=device)
state_dict = ckpt.get('model', ckpt)
clean = OrderedDict((k.replace('module.', ''), v) for k, v in state_dict.items())
model.load_state_dict(clean, strict=False)
model = model.to(device)
model.eval()
print(f'Loaded: {ckpt_file}')

In [ ]:
_, _, test_loader, _, _ = get_dataloaders_test(
    h5_file       = RRUFF_H5,
    batch_size    = BATCH_SIZE,
    num_classes   = NUM_CLASSES,
    num_workers   = NUM_WORKERS,
    prefetch_factor = 2,
    start_col     = START_COL,
    end_col       = END_COL,
    label_mode    = 'categorical',
)
print(f'Test samples: {len(test_loader.dataset)}')

In [ ]:
top1_correct = top3_correct = top5_correct = total = 0

with torch.no_grad():
    for inputs, targets in tqdm(test_loader, desc='Evaluating'):
        inputs  = inputs.to(device)
        targets = targets.to(device)
        outputs = model(inputs)               # (B, 99)
        B = targets.size(0)
        total += B

        # top-k: targets in top-k predictions?
        _, topk = outputs.topk(5, dim=1)      # (B, 5)
        t = targets.view(-1, 1)               # (B, 1)
        top1_correct += topk[:, :1].eq(t).any(1).sum().item()
        top3_correct += topk[:, :3].eq(t).any(1).sum().item()
        top5_correct += topk[:, :5].eq(t).any(1).sum().item()

print(f'\nResults for {RUN_ID} on {Path(RRUFF_H5).name}')
print(f'  Total samples : {total}')
print(f'  Top-1         : {100 * top1_correct / total:.2f}%')
print(f'  Top-3         : {100 * top3_correct / total:.2f}%')
print(f'  Top-5         : {100 * top5_correct / total:.2f}%')